<img src=../figures/Brown_logo.svg width=50%>

## Data-Driven Design & Analyses of Structures & Materials (3dasm)

## Lecture 19.1

### Miguel A. Bessa | <a href = "mailto: miguel_bessa@brown.edu">miguel_bessa@brown.edu</a>  | Associate Professor

### Elvis Aguero | <a href = "mailto: elvis_vera@brown.edu">elvis_vera@brown.edu</a>  | PhD candidate


**What:** A lecture of the "3dasm" course

**Where:** This notebook comes from this [repository](https://github.com/bessagroup/3dasm_course)

**Reference for entire course:** Murphy, Kevin P. *Probabilistic machine learning: an
introduction*. MIT press, 2022. Available online [here](https://probml.github.io/pml-book/book1.html)

**How:** We try to follow Murphy's book closely, but the sequence of Chapters and Sections is
different. The intention is to use notebooks as an introduction to the topic and Murphy's book
as a resource.
* If working offline: Go through this notebook and read the book.
* If attending class in person: listen to me (!) but also go through the notebook in your laptop at the same time. Read the book.
* If attending lectures remotely: listen to me (!) via Zoom and (ideally) use two screens where you have the notebook open in 1 screen and you see the lectures on the other. Read the book.

This is the first of two lectures on **adda**. Today: how you build such a framework, and what
we learned building it. Lecture 19.2: what happened when we pointed it at a real problem.

## **OPTION 1**. Run this notebook **locally in your computer**:
1. Confirm that you have the '3dasm' mamba (or conda) environment (see Lecture 1).
2. Go to the 3dasm_course folder in your computer and pull the last updates of the [repository](https://github.com/bessagroup/3dasm_course):
```
git pull
```
    - Note: if you can't pull the repo due to conflicts (and you can't handle these conflicts), use this command (with **caution**!) and your repo becomes the same as the one online:
```
git reset --hard origin/main
```
3. Open command window and load jupyter notebook (it will open in your internet browser):
```
jupyter notebook
```
5. Open notebook of this Lecture and choose the '3dasm' kernel.

## **OPTION 2**. Use **Google's Colab** (no installation required, but times out if idle):

1. go to https://colab.research.google.com
2. login
3. File > Open notebook
4. click on Github (no need to login or authorize anything)
5. paste the git link: https://github.com/bessagroup/3dasm_course
6. click search and then click on the notebook for this Lecture.

In [1]:
# Basic plotting tools needed in Python.

import matplotlib.pyplot as plt # import plotting tools to create figures
import numpy as np # import numpy to handle a lot of things!

%config InlineBackend.figure_format = "retina" # render higher resolution images in the notebook
plt.rcParams["figure.figsize"] = (8,4) # rescale figure size appropriately for slides

# To limit the number of rows to show in a dataframe, for presentation purposes:
import pandas as pd

pd.set_option('display.max_rows', 10)

In [2]:
# In Google Colab you need to install f3dasm first (locally it is already in the '3dasm'
# environment). Uncomment the line below if you are running in Colab:

# %pip install f3dasm

from f3dasm import ExperimentData   # the same object you used in Lectures 17, 18 and 19

## Outline for today

* What an agent is: tools, nodes, a graph
* Why more than one node: safety, specialization, efficiency
* Where the state lives
* What admits a claim
* Getting started, and one lesson from building it

**Reading material**: this notebook + the
[a3dasm documentation](https://elvis-aguero.github.io/a3dasm/).

Set the frame in the first minute: this is the builder's view, and 19.2 is what happened
when we pointed it at a real problem. Nobody here has to build one to get something out of it.

Two names, and they are not interchangeable. adda is the framework, agentic data-driven design
and analysis. a3dasm is the package you import, and it is what the documentation link points at.

If someone asks whether this is public: the package and its docs are. The campaign in 19.2 is
not yet published.


## Agents, tools, and jobs


* An LLM with a set of tools, and a job is an **agent** — a **subagent** when another agent spawns it. As of 2026 they are dispatched automatically by the coding agents you already use:

<img src=../figures/agent_products.svg width=86%>

* A fixed code path that calls a model at each step is a **workflow**. We will see how to **orchestrate** multiple of them.

Keep this to a minute. The definition is doing the work here, not the product logos.

The question that follows is how an agent knows which tools it has. In a3dasm the tool catalog
is generated from the live tool set and appended to the end of the system prompt at every
invocation, in _src/backends/claude.py, so what an agent reads can never drift from what it can
actually call.

Subagent just means an agent that another agent started. Nothing deeper is implied.


## Six multi-agent architectures

There are multiple ways to arrange multiple agents to collaborate. Some people are trying single agents with tools, networks, supervisors, supervisors-as-tools, hierarchies, and custom graphs.

A graph architecture where each node is a possibly different agent, and edges dictate who can talk to who.

<p align="center"><img src=../figures/agentic_patterns.png width=58%></p>

<sub>Taxonomy from the LangGraph multi-agent documentation; figure from *Agentic patterns: architectures for coordinated AI systems*, Medium.</sub>

We will show you the decisions involved when architecting a _graph_ of agents.

The six panels are someone else's taxonomy and someone else's vocabulary. Say that, credit it,
and move on quickly.

Ours is none of the six exactly. _src/agents/_graphs.py defines five nodes and six directed
edges: the strategizer reaches all four others, and the data generator and the implementer each
have one shortcut to the literature reviewer. A debugger exists in the codebase and is
deliberately not in the default graph.

If asked which pattern we are: a supervisor, with two shortcuts.


## Why a graph of agents

Three principles: Safety, specialization, efficiency. 

We will see that there are a few advantages of arranging agents in a graph format.

## Safety I: independence needs a fresh context

An agent asked to check its own work usually agrees with it.

A second agent, started fresh and shown only the result, gives you a real second opinion.

The objection you should expect is that a second model is just as biased as the first. Take it
seriously rather than brushing it off.

The answer is not that the critic is smarter. It starts from a fresh context, it sees the
artifact rather than the reasoning that produced it, and its verdict has consequences: three
non-PASS verdicts and the run closes stamped UNGATED instead of pretending it passed. That count
is deliberately hidden from the agent so it cannot learn to exhaust the critic.

This lowers the failure rate. It does not remove it.


## Safety II: restricted tools per node

Each node gets a specialized set of tools tailored to their goals.

Nineteen words on the slide, so this one is entirely yours. Give one concrete example.

Every agent class declares a frozen set of tool names, and the catalog at the end of its prompt
is built from exactly that set. The critic's is read-only: it can read the record and the
deliverable and it cannot write either. The implementer is the only node that calls the
evaluator.

That asymmetry is the whole point. A node that cannot write the record cannot quietly fix a
result it dislikes.


## Specialization, one model per node

Different jobs, different models. A strong model plans, a cheaper one executes.

A fine-tuned open-weights model already is on par with frontier models on narrow tasks at a fraction of the compute. Each node can get its own backend with their own settings.

Be careful here, because the slide promises more than the default actually ships.

The mechanism is real: _src/agent_runtime.py resolves a node's model as its own setting or else
the run's, so any node can take a different one. But no shipped agent sets it, so in the default
graph all five nodes run the same model. The one mixed run we have used a stronger strategizer
with cheaper workers, and it was configured by hand.

If asked whether we benchmarked the open-weights claim: we have not. That is headroom, not a
result.


## Efficiency through parallel, cancellable work

Work is handed over as a **delegation**: a unit you can cancel, retry, or abandon. A six-hour run that dies at hour five does not start again.

Independent units also run at the same time.

<p align="center"><img src=../figures/hypothesis_delegation.png width=78%></p>


Connect this forward. The _delegation_id column on the next code slide is this same object,
stamped onto every row the delegation produced.

One correction worth having ready, because the figure invites it. A delegation is not a hand-off
to another graph node. Delegate() copies the target agent and runs it in a background thread
inside the calling node, so several are genuinely in flight at once and the planner is
re-prompted as reports arrive.

Cancellable means the unit is abandoned. It does not mean a running solver is killed mid-solve.


## When one chat is enough

Short task. Cheap to check. You already know the plan.

Then a single chat is the right tool, and a graph is overhead.

## The shared state is a file on disk

A node is stateless between calls.

Agent frameworks typically checkpoint the whole conversation, so a thread can be resumed.
We chose to make the record the state instead: that buys reproducibility rather than
resumable dialogue: every claim in the deliverable recomputes from the record, so a run
is auditable by someone who never saw the conversation.

This is the slide the whole framework rests on. Do not rush it.

The trade is real and worth stating as a trade: we gave up resumable dialogue. What we bought is
that a run can be audited by someone who never saw the conversation, because every number in the
deliverable is recomputed from experiment_data/ rather than quoted from a transcript.

If someone asks how a run knows it is finished, that is not the state's job. A run closes only
through an accepted Done(), and there are nine distinct ways one can end. 19.2 covers them.


In [3]:
# A record from a real run: 538 evaluations of a lattice design.
ledger = ExperimentData.from_file('.')
design, result = ledger.to_pandas()
record = design.join(result)

# the design columns are your final project's variables; the _ columns are the stamps
record[["ratio_a", "ratio_pitch", "coilable", "sigma_crit",
        "_delegation_id", "_source", "_wall_ms"]].head(6)

,ratio_a,ratio_pitch,coilable,sigma_crit,_delegation_id,_source,_wall_ms
0,0.022644,0.680000,1,4.046822,D000,supercompressible-material,70499.483
1,0.009213,0.681277,1,0.807276,D000,supercompressible-material,239986.841
2,0.009213,0.681277,0,NaN,D000,supercompressible-material,142069.682
3,0.009213,0.681277,0,3.245657,D000,supercompressible-material,141805.785
4,0.026858,1.317206,0,21.218537,D000,supercompressible-material,141822.288
5,0.009213,0.681277,0,3.245657,D000,supercompressible-material,141473.844


Do not tab through this dataframe live. Two columns that are not selected here hold absolute
cluster paths, including a scratch directory and a username. The seven columns shown are safe.

The provenance: 538 evaluations from the supercompressible metamaterial study, the same problem
as 19.2, which is what _source records. ratio_a and ratio_pitch are geometry ratios; coilable is
the screen for whether a design coils rather than collapsing.

Row 2 earns its place. It was screened out, and no result was recorded for it.


## Choosing your own topology

Ours has one node that plans and delegates to the others. That is an accident of our problem,
not a principle: it is a bottleneck and a single point of failure.

Your problem, your graph.

Name the node the slide leaves anonymous: the planner is the strategizer, and it is also the
graph's entry point.

The five in the ratified topology are the strategizer, the literature reviewer, the data
generator, the implementer and the critic. The admission on the slide is genuine, not modesty. A
hub is a bottleneck and a single point of failure, and we would not defend it as a principle.

If asked what we would change: give the workers edges to each other, so a result need not travel
through the planner to be used.


## Popper's advice

As of 2026, LLMs are generalists by default.

That means they tend to output reasonable sounding answers, that tend to regress to the mean, and the whole user-agent interaction is probabilistic by nature. 

We try to circumvent that by emphasizing the rules of science to all agents in the workflow. A whole deal of research has been done in the lines of philosophy of science, and we might start by delimiting the Popperian rules of science. 


Slow down here. This is the pivot from engineering to epistemics and the room may not expect a
mechanics lecture to make it.

The move that matters is that the rules are not requested politely at runtime. They are compiled
into the system prompt of every node that judges a claim, identically, so that two agents who
disagree cite the same numbered clause instead of negotiating what falsification means.

If someone objects that Popper is contested: agreed, and we are not claiming otherwise. We
needed a rule that stops a model confirming itself.


### Science rule 1: one falsifiable claim

One file, quoted verbatim into every node that judges a claim. §1:

> A hypothesis is ONE falsifiable claim carrying a registered prediction: the observable
> whose occurrence would refute the claim.

Register what would refute you, before you look.


Name the file, because the slide does not. It is FALSIFICATION_CHARTER in
a3dasm/_src/knowledge/charter.py, six numbered clauses, compiled into the strategizer's and the
critic's prompts when those classes are defined.

Only two of the six are quoted in this lecture. This one is the definition; the next slide has
section 5.

Register means written down before you look. If asked who enforces it, the answer is the critic:
a hypothesis with no registered prediction does not get a closing verdict.


### Science rule 2: corroboration, not proof

§5:

> SUPPORTED [...] means the hypothesis survived at least one adequate attempt to refute
> it. You never "confirm" a hypothesis; you only fail to falsify it.

Four statuses, and only four: `OPEN`, `SUPPORTED`, `FALSIFIED`, `INCONCLUSIVE`.


The four statuses matter more than they look, and 19.2 turns on the difference between two of
them.

FALSIFIED means an adequate test contradicted the registered prediction. INCONCLUSIVE means the
test was not adequate, so the claim survives untested. A contradiction from a flawed test
indicts the test, not the claim.

19.2 has the worked example, and it is ours: our own favourite idea was marked INCONCLUSIVE
while the evidence licensed nothing stronger, then FALSIFIED later when thirty designs failed to
coil.


## The reproduction gate

The deliverable is a notebook. Its code cells recompute the result from the record.

It is executed in a clean sandbox before the run may close.


Say the number, because it is the part people remember. The deliverable gets six attempts to
reproduce itself, and if it still will not run the run is stamped FAILED and the critic is never
spent on it.

Clean means a fresh process that never saw the run: no variables in memory, no cached state, only
the record on disk. That is what makes the notebook evidence instead of a transcript.

The honest limit: reproducing a number checks the arithmetic, not the physics. The charter and the
critic are what address the physics.


## Simplicity above all: The study folder

```
my_study/
  PROBLEM_STATEMENT.md   # required: the brief
  config.yaml            # optional: model, budget, how a design is scored
  workspace/
    evaluator.py         # optional: your ground truth
```

Keep this brisk. It is orientation rather than argument.

Only the problem statement is required. Everything else has a default: no config file means the
default model and no budget, and no evaluator means the system has to construct one, which is
the data generator's job and the reason that node exists.

If asked where the results go, they land in experiment_data/ under the study directory, which is
exactly the folder you loaded from two slides ago.


## The problem statement file

```markdown
# Minimise a 2-D quadratic

## Objective
Minimise y = (x1 - 1)^2 + (x2 + 2)^2.

## Design space
| variable | type | bounds | units |
|---|---|---|---|
| x1 | continuous | [-5, 5] | dimensionless |
| x2 | continuous | [-5, 5] | dimensionless |
```

Objective, bounds, units. That table is a `Domain`, written in prose.

## Model, budget, and evaluator

```yaml
model: haiku
eval_budget: 200
evaluator:
  entrypoint: "workspace/evaluator.py:evaluate"
  output_names: [y]
```

```python
def evaluate(x1: float, x2: float) -> float:
    return (x1 - 1.0) ** 2 + (x2 + 2.0) ** 2
```

This is the slide people photograph, so give it the provenance it deserves.

The model string is not a friendly name. The default is claude-haiku-4-5-20251001, set in
a3dasm/_src/agent_runtime.py, and resolution runs explicit argument, then this file, then a
backend default, which is qwen2.5:1.5b against a local ollama server.

There is an alias table, but only on the open-weights path, where gemma-4 expands to a full
HuggingFace id. A short name like haiku does not expand: it is passed through and fails.

eval_budget is advisory. It warns, and it never stops a run.


## One call, one notebook

```python
from a3dasm import AgenticRun
report = AgenticRun(study_dir="my_study").execute()
```

Back comes `pipeline.ipynb`, the record of every evaluation, and whether the run passed
its gate.

Expect three questions the moment this appears: what it costs, how long it takes, and whether a
key is needed.

Costs belong to 19.2, where a real campaign is tens of dollars and a few hours. You need
credentials for whichever backend you pick; four are registered and two of them are local, so an
ollama or vLLM server on your own hardware is a supported path rather than a workaround.

A run halts on a spend ceiling, on twelve consecutive errors from one target, or on a time
backstop. Those are the hard stops. The budgets are not.


## An agent's account of a failure

Every node writes a retrospective when a run closes: what it found hard, what blocked it, where it
contradicted itself. It is the highest-signal artifact we have, and it can still be wrong about
mechanism.

One run's retrospective blamed a silent crash for a 63% evaluation failure rate, and pointed
at the licence server saturating at 16-way concurrency.

Deliver the 63% as a quotation, not as a finding. We cannot reproduce it from the record and we
do not know what it was measured over.

Resist explaining that here. The next slide is where it pays off.

Retrospectives are first-person entries a node writes as the run closes, kept in
debug/retrospectives.jsonl. They are the highest-signal artifact we have for what went wrong,
and this one was wrong about the mechanism and about the rate. A lead, not evidence.


In [4]:
# What the run reported, against what the record shows.
d = design.join(result)
attempted = d[d.coilable == 1]              # passed the coilability screen
failed = ~attempted.riks_converged.astype(bool)

print("solves that never converged:", int(failed.sum()), "of", len(attempted))
print("failure rate:", round(100 * failed.mean(), 1), "%")
print("median wall time, failed vs converged:",
      int(attempted.loc[failed, '_wall_ms'].median() / 1000), "s vs",
      int(attempted.loc[~failed, '_wall_ms'].median() / 1000), "s")

solves that never converged: 154 of 469
failure rate: 32.8 %
median wall time, failed vs converged: 329 s vs 66 s


The retrospective is a lead. The record is the diagnosis: these solves ran five times longer than
the ones that worked before they died, which is a wait, not a crash. The 63% is not in the record either: the rate here is 32.8%.

Read the numbers off the screen rather than from memory: 154 of 469 attempted solves never
converged, and the failures ran a median 329 seconds against 66 for the ones that worked.

The inference to defend is the last one. Long-then-dead is a queue or a licence wait; a crash is
short-then-dead. That is an argument from timing, not a measurement of the licence server, and it
is worth conceding the difference.

The other 69 of the 538 never passed the screen, so nothing was ever solved for them.


## Summary

* An agentic workflow is a graph of nodes, each a model with tools, choosing its own next step.
* Split the work for **safety**, **specialization** and **efficiency**, and not otherwise.
* The state is the data, not the conversation.
* A claim is admitted by surviving refutation and by reproducing from the record.
* One folder in, one notebook out.

Land the three principles and stop. The audience does not need the hour replayed.

If you want one honest sentence about what we would build differently, it is the hub. Every
result travels through the strategizer, which makes it both the bottleneck and the single point
of failure, and the workers should have had edges to each other.

Then hand over to 19.2: the same machinery pointed at a real metamaterial, including the run
where it caught its own headline and killed it.


## Next lecture

The same framework, pointed at a real design problem, and what it actually found.

### See you next class

Have fun!